# Evaluate baseline vs proposed (soft-voting) on Kaggle
Session options: **GPU T4 x2**, **Internet On**. Attach two private datasets with **Add Input**:
`hogformer-baseline-final` (contains `net_g_latest.pth` of the baseline) and `hogformer-softvote-final` (same for the proposed model).
Both models are scored on the SAME images: all of Test1 (750) and RainDrop test_a (58), plus a fixed random subset of Snow100K-S / -L (seeded, identical for both models).

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
%cd /kaggle/working
!test -d HOGformer_DLCV || git clone -b claude/lucid-fermi-emhnbo https://github.com/adityaj1706/HOGformer_DLCV.git
%cd /kaggle/working/HOGformer_DLCV
!git pull
!git log -1 --oneline
!pip install -q -r settingI/requirements_colab.txt

In [ ]:
# Test data only (skips the 11 GB training set): Test1, RainDrop test_a, Snow100K-S/L  (~10-15 min)
!mkdir -p /tmp/data /content
!ln -sfn /tmp/data /content/data
%cd /kaggle/working/HOGformer_DLCV
!SKIP_TRAIN=1 bash scripts/download_data.sh /tmp/data

In [ ]:
# Find the two checkpoints
import glob
def find(tag):
    c = [p for p in glob.glob('/kaggle/input/**/net_g_latest.pth', recursive=True) if tag in p]
    assert len(c) == 1, f'expected exactly one net_g_latest.pth whose path contains {tag!r}, found {c}'
    return c[0]
BASE_W, SOFT_W = find('baseline'), find('softvote')
print(BASE_W); print(SOFT_W)

In [ ]:
# Run both evaluations in the BACKGROUND (about 1-2 h in total; adjust --snow_n to trade time for precision)
import subprocess, os
os.chdir('/kaggle/working/HOGformer_DLCV/settingI')
cmd = (f"python -u evaluate.py --config Allweather/Options/Allweather_HOGformer_baseline.yml --weights {BASE_W} --out /kaggle/working/results_baseline.json --name baseline --snow_n 300 && "
       f"python -u evaluate.py --config Allweather/Options/Allweather_HOGformer_softvote.yml --weights {SOFT_W} --out /kaggle/working/results_softvote.json --name softvote --snow_n 300")
proc = subprocess.Popen(f'({cmd}) > /kaggle/working/eval_stdout.log 2>&1', shell=True)   # parentheses: log BOTH runs
print('started, pid', proc.pid)

In [ ]:
# MONITOR (run any time)
!tail -n 8 /kaggle/working/eval_stdout.log | cut -c1-200
print('still running' if proc.poll() is None else f'FINISHED, exit code {proc.returncode}')

In [ ]:
# WAIT until finished. Prints the latest evaluation line every 5 minutes (visible in the Logs tab).
import time, subprocess
while proc.poll() is None:
    time.sleep(300)
    r = subprocess.run("tail -n 3 /kaggle/working/eval_stdout.log", shell=True, capture_output=True, text=True).stdout.strip()
    print(time.strftime('%H:%M:%S'), r[-300:], flush=True)
print('finished, exit code', proc.returncode)

In [ ]:
# Paired comparison: table with 95% confidence intervals
%cd /kaggle/working/HOGformer_DLCV/settingI
!python compare_results.py /kaggle/working/results_baseline.json /kaggle/working/results_softvote.json --md /kaggle/working/comparison.md